# Embodied Agents and Robotics

## Scenario: place a warehouse package safely

This concise module introduces VLA systems, navigation, manipulation, embodied planning, physical feedback, simulation, and safety. The lab is deterministic and controls no hardware.

**Rule:** physical action is complete only after sensor verification—not after a model emits a command.

![Embodied control loop](../../../assets/embodied-agents-loop.svg)

Use the loop in small, verifiable increments. Failed verification or safety uncertainty produces a safe stop and human escalation.

## 1. VLA and embodied planning

Vision-language-action models combine visual observation and task language to produce action representations. They help generalize semantic instructions, but do not replace robot control, collision checking, force limits, or a physical safety supervisor. A robust plan decomposes `place package` into navigation, pose verification, bounded grasp, grasp verification, placement, and placement verification.

In [ ]:
"""Safe, deterministic embodied-agent simulation; it controls no physical hardware."""
from dataclasses import dataclass
@dataclass
class RobotState: object_seen:bool=True; clear_path:bool=True; force_newtons:float=0.0; holding:bool=False; events:list[str]=None
def run_demo():
 s=RobotState(events=[]); s.events += ["sense: red bin detected", "plan: navigate then pick", "safety: workspace-clear"]
 if not s.clear_path or s.force_newtons>10: s.events.append("stop-and-escalate"); return s
 s.events += ["act: bounded navigation", "observe: at bin", "act: low-force grasp", "observe: grasp verified"]; s.holding=True; assert s.holding; return s
if __name__=="__main__": print("\n".join(run_demo().events))


In [1]:
from lab import run_demo, RobotState

state = run_demo()
print(*state.events, sep='\n')
assert state.holding

sense: red bin detected
plan: navigate then pick
safety: workspace-clear
act: bounded navigation
observe: at bin
act: low-force grasp
observe: grasp verified


## 2. Simulation, feedback, and safety

Use simulation/digital twins to randomize lighting, objects, friction, sensor noise, and task variation. Before deployment, test collisions, contact force, success, recovery, latency, and out-of-distribution conditions. On hardware, independently enforce speed, force, workspace, proximity, emergency stop, freshness, and confirmation constraints.

**Failure experiment:** an obstructed path or high force must prevent action, hold a safe state, log the observation, and ask for human intervention.

In [2]:
unsafe = RobotState(clear_path=False, events=[])
# Equivalent policy branch: never navigate when perception says the path is blocked.
if not unsafe.clear_path or unsafe.force_newtons > 10:
    unsafe.events.append('stop-and-escalate')
print(unsafe.events)
assert unsafe.events == ['stop-and-escalate']

['stop-and-escalate']


## Exercises and references

1. Add confidence/freshness checks before navigation. 2. Add a geofence and restricted-zone approval. 3. Record action/observation traces and define a recovery budget. 4. Define separate simulation and hardware release criteria.

- [RT-2](https://deepmind.google/blog/rt-2-new-model-translates-vision-and-language-into-action/) · [OpenVLA](https://arxiv.org/abs/2406.09246) · [Gemini Robotics](https://deepmind.google/models/gemini-robotics/)
- [VLA survey](https://arxiv.org/abs/2508.15201) · [simulation survey](https://arxiv.org/abs/2505.01458) · [VLA safety](https://arxiv.org/abs/2604.23775)